# 05. Signals, categories and risk

This notebook reproduces the checks of `docs/07_risk_methodology.md` and shows the full analysis of the demonstration messages. The signal check reads only the training part of the corpus. The demonstration messages were written by the project and are synthetic.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def run_tests(*files):
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", *files], cwd=ROOT, capture_output=True, text=True)
    print(result.stdout.strip().splitlines()[-1])

## 1. Unit tests

In [2]:
run_tests("tests/test_signals.py", "tests/test_risk.py", "tests/test_analyzer.py", "tests/test_api.py")

49 passed, 1 warning in 3.83s


## 2. Share of training messages that trigger each signal

In [3]:
from hatari.analysis.eda import load_training
from hatari.analysis.signal_check import REVISION_THRESHOLD, trigger_rates

rates = trigger_rates(load_training())
display(rates.round(3))
print("signals above", REVISION_THRESHOLD, "on legitimate English messages:",
      rates.index[rates["en_legitimate"] > REVISION_THRESHOLD].tolist() or "none")

,en_legitimate,en_suspicious,fr_legitimate,fr_suspicious
money_request,0.002,0.063,0.000,0.143
credential_request,0.000,0.002,0.000,0.029
account_threat,0.000,0.005,0.000,0.057
shortened_link,0.000,0.007,0.000,0.000
mistaken_transfer,0.000,0.001,0.000,0.000
prize_promise,0.001,0.189,0.000,0.071
urgency,0.012,0.118,0.021,0.271
mobile_money,0.000,0.000,0.000,0.000
job_offer,0.002,0.000,0.000,0.000
investment_promise,0.000,0.002,0.000,0.071


signals above 0.05 on legitimate English messages: none


**Findings**

- No signal fires on more than 1.2% of the legitimate English training messages.
- The Mobile Money signal never fires, because the training data contain no Mobile Money fraud. It can only be checked on hand-written examples.

## 3. Demonstration messages

In [4]:
from hatari.demo.examples import EXAMPLES
from hatari.service.analyzer import Analyzer

analyzer = Analyzer()
rows = []
for example in EXAMPLES:
    result = analyzer.analyze(example.text)
    rows.append({"example": example.title, "kind": example.kind, "score": result.risk.score, "level": result.risk.level,
                 "model": result.model.score, "model_points": result.risk.model_points,
                 "signal_points": result.risk.signal_points,
                 "categories": ", ".join(category["label"] for category in result.categories),
                 "signals": ", ".join(signal.id for signal in result.signals)})
display(pd.DataFrame(rows))

,example,kind,score,level,model,model_points,signal_points,categories,signals
0,Gain contre frais,Fraude évidente,95,high,0.992,49.6,45,"Arnaque aux frais avancés, Manipulation psychologique","money_request, prize_promise, urgency, amount"
1,Faux service client,Fraude moins évidente,99,critical,0.986,49.3,50,"Vol de codes ou d'identifiants, Arnaque au Mobile Money, Hameçonnage, Usurpation d'identité possible","credential_request, account_threat, mobile_money, institution"
2,Transfert par erreur,Fraude moins évidente,83,critical,0.653,32.6,50,Arnaque au Mobile Money,"money_request, mistaken_transfer, mobile_money, amount"
3,Colis en attente,"Fraude évidente, en anglais",98,critical,0.950,47.5,50,"Lien suspect, Manipulation psychologique","money_request, shortened_link, urgency, link, amount"
4,Rendez-vous,Message légitime,1,low,0.029,1.5,0,Non classé,
5,Notification de dépôt,Message ambigu,54,high,0.986,49.3,5,Non classé,amount
6,Rappel de facture,Message ambigu,52,high,0.939,46.9,5,Non classé,amount


**Findings**

- The four fraudulent examples reach a high or critical risk, and the appointment message a low risk.
- The deposit notification and the bill reminder, plausibly legitimate, reach a high risk through the model alone, with no strong signal. This is the bias towards formal French, shown on purpose during the demonstration.

## 4. One complete analysis

In [5]:
result = analyzer.analyze(EXAMPLES[1].text)
print(EXAMPLES[1].text, "\n")
for signal in result.signals:
    print(f"- {signal.label} ({signal.strength}): {' | '.join(signal.evidence)}")
print()
for sentence in result.explanation:
    print(sentence)
print()
for advice in result.recommendations:
    print("*", advice)
print()
for warning in result.warnings:
    print("!", warning)

Bonjour, ici le service client. Suite à une mise à jour de votre compte Orange Money, merci de nous communiquer le code que vous allez recevoir par SMS afin d'éviter la suspension de votre ligne. 

- Demande de code, de PIN ou de mot de passe (strong): communiquer le code | code que vous allez recevoir par SMS
- Menace sur un compte (strong): suspension de votre ligne
- Contexte Mobile Money (medium): Orange Money
- Se présente comme une institution (medium): service client | Orange Money

Signaux observés dans le texte : demande de code, de pin ou de mot de passe, menace sur un compte, contexte mobile money, se présente comme une institution.
Modèle d'apprentissage automatique : score de 0,99 sur 1. La formulation ressemble davantage aux messages suspects des données d'entraînement.
Évaluation finale : 99 sur 100, dont 49 points venant du modèle et 50 points venant des signaux. Ce score classe le niveau d'inquiétude, ce n'est pas une probabilité de fraude.

* Ne communiquez jamais un 